In [ ]:
!pip install wikipedia

In [ ]:
import nltk
import re
import requests
from bs4 import BeautifulSoup
from collections import Counter
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from nltk.tag import UnigramTagger, DefaultTagger
from nltk.chunk import RegexpParser

# Download dos recursos (incluindo punkt_tab)
print("Baixando recursos do NLTK...")
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('mac_morpho')
print("Recursos baixados!\n")

Baixando recursos do NLTK...
Recursos baixados!



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package mac_morpho to /root/nltk_data...
[nltk_data]   Package mac_morpho is already up-to-date!


# 1. EXTRAÇÃO

In [ ]:
def extrair_texto_wikipedia(url):
    """Extrai o texto usando requests com User-Agent"""
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')
        content_div = soup.find('div', {'id': 'mw-content-text'})

        if content_div:
            paragraphs = content_div.find_all('p')
            texto = ' '.join([p.get_text() for p in paragraphs])
            return texto
        else:
            return None

    except Exception as e:
        print(f"Erro ao extrair texto: {e}")
        return None

# Extrai o texto
url = "https://pt.wikipedia.org/wiki/Aprendizagem_profunda"
texto_original = extrair_texto_wikipedia(url)

if texto_original:
    print(f"✓ Texto extraído com sucesso! Tamanho: {len(texto_original)} caracteres\n")
    print("Primeiros 500 caracteres:")
    print(texto_original[:500] + "...\n")
else:
    print("✗ Falha na extração do texto")
    exit()

✓ Texto extraído com sucesso! Tamanho: 64244 caracteres

Primeiros 500 caracteres:
No aprendizado de máquina, a aprendizagem profunda se concentra na utilização de redes neurais multicamadas para executar tarefas como classificação, regressão e aprendizagem de representação. O campo se inspira na neurociência biológica e é centrado em empilhar neurônios artificiais em camadas e "treiná-los" para processar dados. O adjetivo "profunda" se refere ao uso de múltiplas camadas (variando de três a várias centenas ou milhares) na rede. Os métodos usados podem ser supervisionados, semi...



# 2. TOKENIZAÇÃO E LIMPEZA

In [ ]:
def limpar_texto(texto):
    """Remove pontuações, converte para minúsculo e remove stopwords"""

    # Regex para manter letras acentuadas, números e espaços
    texto_sem_pontuacao = re.sub(r'[^\w\sáéíóúâêôãõçÁÉÍÓÚÂÊÔÃÕÇ]', ' ', texto)

    # Tokenização (agora com punkt_tab instalado)
    tokens = nltk.word_tokenize(texto_sem_pontuacao, language='portuguese')

    # Minúsculo
    tokens = [token.lower() for token in tokens]

    # Remove stopwords
    stop_words = set(stopwords.words('portuguese'))
    tokens_limpos = [token for token in tokens
                     if token not in stop_words and len(token) > 1]

    return tokens_limpos

tokens_limpos = limpar_texto(texto_original)
print(f"✓ Tokenização concluída! Total de tokens após limpeza: {len(tokens_limpos)}")
print(f"Primeiros 20 tokens: {tokens_limpos[:20]}\n")

✓ Tokenização concluída! Total de tokens após limpeza: 6031
Primeiros 20 tokens: ['aprendizado', 'máquina', 'aprendizagem', 'profunda', 'concentra', 'utilização', 'redes', 'neurais', 'multicamadas', 'executar', 'tarefas', 'classificação', 'regressão', 'aprendizagem', 'representação', 'campo', 'inspira', 'neurociência', 'biológica', 'centrado']



# 3. ANÁLISE ESTATÍSTICA

In [ ]:
# 3.1 Tokens mais frequentes
freq_dist = FreqDist(tokens_limpos)
tokens_mais_frequentes = freq_dist.most_common(10)

print("=" * 50)
print("10 TOKENS MAIS FREQUENTES")
print("=" * 50)
for i, (token, freq) in enumerate(tokens_mais_frequentes, 1):
    print(f"{i:2d}. '{token}': {freq} ocorrências")
print()

# 3.2 Bigramas mais comuns
bigramas = list(nltk.bigrams(tokens_limpos))
freq_bigramas = FreqDist(bigramas)
bigramas_mais_comuns = freq_bigramas.most_common(5)

print("=" * 50)
print("5 BIGRAMAS MAIS COMUNS")
print("=" * 50)
for i, ((p1, p2), freq) in enumerate(bigramas_mais_comuns, 1):
    print(f"{i:2d}. '{p1} {p2}': {freq} ocorrências")
print()

10 TOKENS MAIS FREQUENTES
 1. 'redes': 115 ocorrências
 2. 'profunda': 95 ocorrências
 3. 'neurais': 95 ocorrências
 4. 'aprendizagem': 90 ocorrências
 5. 'rede': 72 ocorrências
 6. 'neural': 53 ocorrências
 7. 'dados': 49 ocorrências
 8. 'profundas': 43 ocorrências
 9. 'reconhecimento': 41 ocorrências
10. 'camadas': 40 ocorrências

5 BIGRAMAS MAIS COMUNS
 1. 'redes neurais': 91 ocorrências
 2. 'aprendizagem profunda': 78 ocorrências
 3. 'rede neural': 43 ocorrências
 4. 'neurais profundas': 28 ocorrências
 5. 'reconhecimento fala': 16 ocorrências



# 4. POS TAGGING (Mac-Morpho)

In [ ]:
def treinar_tagger():
    """Treina UnigramTagger com mac_morpho e fallback DefaultTagger('N')"""
    from nltk.corpus import mac_morpho

    tagged_sents = mac_morpho.tagged_sents()
    split = int(0.8 * len(tagged_sents))
    train_sents = tagged_sents[:split]

    default_tagger = DefaultTagger('N')
    unigram_tagger = UnigramTagger(train_sents, backoff=default_tagger)
    return unigram_tagger

print("Treinando o tagger...")
tagger = treinar_tagger()
print("✓ Tagger treinado com sucesso!\n")

# Aplica POS tagging nos primeiros 100 tokens
tokens_exemplo = tokens_limpos[:100]
pos_tagged = tagger.tag(tokens_exemplo)

print("=" * 50)
print("POS TAGGING (primeiros 20 tokens)")
print("=" * 50)
for i, (token, tag) in enumerate(pos_tagged[:20], 1):
    print(f"{i:2d}. {token:20s} -> {tag}")
print()

Treinando o tagger...
✓ Tagger treinado com sucesso!

POS TAGGING (primeiros 20 tokens)
 1. aprendizado          -> N
 2. máquina              -> N
 3. aprendizagem         -> N
 4. profunda             -> ADJ
 5. concentra            -> V
 6. utilização           -> N
 7. redes                -> N
 8. neurais              -> N
 9. multicamadas         -> N
10. executar             -> V
11. tarefas              -> N
12. classificação        -> N
13. regressão            -> N
14. aprendizagem         -> N
15. representação        -> N
16. campo                -> N
17. inspira              -> V
18. neurociência         -> N
19. biológica            -> N
20. centrado             -> PCP



# 5. EXTRAÇÃO DE GRUPOS NORMAIS (NP Chunking)

In [ ]:
def extrair_grupos_nominais(texto_taggeado):
    """Extrai grupos de 2+ NPROP ou 2+ N sequenciais"""
    gramatica = r"""
        NP: {<NPROP>{2,}}
            {<N>{2,}}
    """
    chunk_parser = RegexpParser(gramatica)
    arvore = chunk_parser.parse(texto_taggeado)

    grupos = []
    for subtree in arvore.subtrees():
        if subtree.label() == 'NP':
            grupo_texto = ' '.join([token for token, tag in subtree.leaves()])
            grupos.append(grupo_texto)
    return grupos

grupos_nominais = extrair_grupos_nominais(pos_tagged)

print("=" * 50)
print("GRUPOS NORMAIS COMPOSTOS (primeiras 5 entidades)")
print("=" * 50)

if grupos_nominais:
    for i, grupo in enumerate(grupos_nominais[:5], 1):
        print(f"{i:2d}. {grupo}")
else:
    print("Nenhum grupo nominal composto encontrado nos primeiros 100 tokens.")
    print("Isso pode acontecer porque o texto precisa de mais tokens.")

GRUPOS NORMAIS COMPOSTOS (primeiras 5 entidades)
 1. aprendizado máquina aprendizagem
 2. utilização redes neurais multicamadas
 3. tarefas classificação regressão aprendizagem representação campo
 4. neurociência biológica
 5. empilhar neurônios
